# Pymoca new flattening and related features — a guided tour

This runnable notebook documents the new and improved capabilities of pymoca. Each
section shows examples of the new behavior, then immediately contrasts with results from
pymoca version 0.11.2, the most recent PyPI release when this notebook was created. The
side-by-side comparison makes the claims in this document machine-checkable: re-execute
the notebook to verify everything still holds. The new version was developed according
to Modelica Language Specification version 3.5. All new features are expected to be
forward-compatible with the current latest version but this has not been verified.

## Sections

1. [New flattening API](#part1)
2. [MODELICAPATH](#part2)
3. [The `pymoca` CLI / `compiler.py`](#part3)
4. [MLS-compliance](#part4)

## Setup

New pymoca runs in the current python environment for this notebook. You will need to create a separate venv with Pymoca 0.11.2 installed in the OLD_PY location below or OLD_PY to something else. There is also a script alongside this notebook to regenerate it when run from the repo.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

from pymoca import ast, parser, tree

# Absolute path to the repo root (this notebook lives in doc/)
REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
if REPO_ROOT.name == "doc":
    REPO_ROOT = REPO_ROOT.parent
MODEL_DIR = REPO_ROOT / "test" / "models"
MSL4_DIR  = REPO_ROOT / "test" / "libraries" / "MSL-4.0.x"

# Python interpreter for pymoca 0.11.2.
# PYTHONPATH must be stripped so the dev-install of new pymoca doesn't shadow it.
OLD_PY = str(REPO_ROOT / ".venv-pymoca-011" / "bin" / "python")
_OLD_ENV = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}

_ansi_re = __import__("re").compile(r"\x1b\[[0-9;]*[mK]")


def run_old(code_str: str, *, timeout: int = 30) -> None:
    """Run *code_str* under pymoca 0.11.2 and print the combined output."""
    result = subprocess.run(
        [OLD_PY, "-c", code_str],
        env=_OLD_ENV,
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    combined = _ansi_re.sub("", result.stdout + result.stderr)
    print(combined.strip() if combined.strip() else "(no output)")

# Confirm versions
import pymoca as _pm
print(f"New pymoca: {_pm.__version__}")
print("Old pymoca:", end=" ")
run_old("import pymoca; print(pymoca.__version__)")

New pymoca: 0.11.2+188.gc9200d54.dirty
Old pymoca: 0.11.2


<a id="part1"></a>
## Part 1 — New flattening API

The table below summarizes the entry points that are new or meaningfully changed.
Detailed live examples follow for each row.

|#| Capability | 0.11.2 | New |
|---|---|---|---|
| 1 | [Parse a file](#sec1-1) | `open()` + `read()` + `parser.parse()` → `ast.Tree` | `parser.parse_file(path)` → `ast.Tree` |
| 2 | [Syntax errors](#sec1-2) | `None` returned, ANTLR noise on stderr | `ModelicaSyntaxError` raised |
| 3 | [Name lookup](#sec1-3) | `ast_class.find_class(comp_ref)` → `ast.Class`| `tree.find_name(ast_class, name)` → `ast.Class` or `ast.Symbol`|
| 4 | [Name lookup errors](#sec1-4) | `None` returned; model flattens with wrong or absent value | `NameLookupError` raised |
| 5 | [Instantiate without flatten](#sec1-5) | `ast_class.find_class(comp_ref)` + `tree.build_instance_tree(ast_class)` → `ast.Class` | `tree.instantiate(ast_tree, name)` → `ast.InstanceClass`|
| 6 | [Flatten Class to Class](#sec1-6) | `tree.flatten_class(ast_class))` → `ast.Class` | `tree.flatten_instance(instance_class)` → `ast.InstanceClass` |
| 7 | [Flatten name from Tree to Class](#sec1-7) | None | `tree.flatten_class(ast_tree, name)` → `ast.InstanceClass` |
| 8 | [Flatten name from Tree to Tree](#sec1-8) | `tree.flatten(ast_tree, comp_ref)` → `ast.Tree` | `tree.flatten(ast_tree, name)` → `ast.Tree` |
| 9 | [Semantic errors](#sec1-9) | self-extends raises a generic error; other violations silently accepted | `ModelicaSemanticError` raised |

The types of the variables above are `path: pathlib.Path`, `comp_ref: ast.ComponentRef`,
`ast_class: ast.Class`, `name: str | ast.ComponentRef`, `ast_tree: ast.Tree`,
`instance_class: ast.InstanceClass`.

<a id="sec1-1"></a>
### 1.1 `parser.parse_file(path)`

`parse_file` is a new convenience function that opens the file, parses it, and
decorates any `ModelicaSyntaxError` with the file path — saving some repeated boilerplate.

Let's parse this simplified Modelica model from the test suite that mirrors a gear model
using the Modelica Standard Library.  

In [2]:
modelica_file = MODEL_DIR / "ConstantInModificationScope.mo"
print(modelica_file.read_text())

package Modelica
  package Mechanics
    package Rotational
      package Interfaces
        partial model PartialFriction
          constant Integer Backward = -1;
          constant Integer Forward = 1;
          constant Integer Unknown = 3;
          Integer mode(final min = Backward, final max = Unknown, start = Unknown, fixed = true);
          Real sa;
        end PartialFriction;

        partial model PartialElementaryTwoFlanges
          Real flange_a;
          Real flange_b;
        end PartialElementaryTwoFlanges;
      end Interfaces;

      package Components
        model BearingFriction
          extends Rotational.Interfaces.PartialElementaryTwoFlanges;
          extends Rotational.Interfaces.PartialFriction;
          Real phi;
          Real w;
        end BearingFriction;
      end Components;
    end Rotational;
  end Mechanics;
end Modelica;

model GearType1
  Modelica.Mechanics.Rotational.Components.BearingFriction bearingFriction;
end GearType1;



In [3]:
# New: parse_file returns an ast.Tree directly from a file name
ast_tree = parser.parse_file(modelica_file)
print("classes:", list(ast_tree.classes))

# parse_file also raises a clear error on missing files
try:
    parser.parse_file("/nonexistent/path/Model.mo")
except FileNotFoundError as exc:
    print(f"\nFileNotFoundError: {exc}")

classes: ['Real', 'Integer', 'Boolean', 'String', 'StateSelect', 'Clock', 'Modelica', 'GearType1']

FileNotFoundError: File not found: /nonexistent/path/Model.mo


**In pymoca 0.11.2:** `parse_file` does not exist. Here is the equivalent.

In [4]:
run_old(f"""
from pymoca import parser
with open("{modelica_file}") as f:
    txt = f.read()
ast_tree = parser.parse(txt)
print("classes:", list(ast_tree.classes))
""")


classes: ['Modelica', 'GearType1']


Comparing this output to the output from the new version of the parser above, you see
that the new one adds built-in classes at the root of the tree per the Modelica Language
Spec.

<a id="sec1-2"></a>
### 1.2 `parser.ModelicaSyntaxError`

The parser now raises `ModelicaSyntaxError` with file, line, and column information
referencing the offending Modelica source code.  Because it inherits from Python's
built-in `SyntaxError`, an uncaught exception prints a Modelica-aware traceback — the
file and offending line appear in the standard Python traceback output with no extra
work by the caller.

The pymoca repo contains a test file with a syntax error documented in the Modelica comment.

In [5]:
bad_src_file = Path(MODEL_DIR) / "RedeclareNestedClass.mo.fail_parse"
print(bad_src_file.read_text())

model A
  Real x;
end A;

model D
  replaceable model C
    replaceable model B
      Real y;
    end B;

    B z;
  end C;

  C c;
end D;

// Redeclaring nested (i.e. "contains dot") component references is not
// allowed (as it would likely horribly complicate things). Parser should
// fail.
model E
  extends D(c.z.x(nominal=2), redeclare model C.B=F);
end E;



If you don't catch the `ModelicaSyntaxError`, Python prints a traceback referencing the Modelica source. Here we catch the exception and print it so Jupyter will continue to run all cells in the notebook without stopping because of an uncaught exception.

In [6]:
import traceback
try:
    parser.parse_file(bad_src_file)
except parser.ModelicaSyntaxError:
    traceback.print_exc()

Traceback (most recent call last):
  File "/Users/rutanwk/Claude/tmp/pymoca/src/pymoca/parser.py", line 1616, in parse_file
    ast_tree = parse(txt, trace=trace)
  File "/Users/rutanwk/Claude/tmp/pymoca/src/pymoca/parser.py", line 1446, in parse
    modelica_file = parse_text(
        txt,
    ...<4 lines>...
        bypass_cache=bypass_cache,
    )
  File "/Users/rutanwk/Claude/tmp/pymoca/src/pymoca/parser.py", line 1479, in parse_text
    return _parse_text(txt, trace=trace)
  File "/Users/rutanwk/Claude/tmp/pymoca/src/pymoca/parser.py", line 1261, in _parse_text
    parse_tree = parser.stored_definition()
  File "/Users/rutanwk/Claude/tmp/pymoca/src/pymoca/generated/ModelicaParser.py", line 706, in stored_definition
    self.stored_definition_class()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/rutanwk/Claude/tmp/pymoca/src/pymoca/generated/ModelicaParser.py", line 770, in stored_definition_class
    self.class_definition()
    ~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/rutanwk/Cl

If you catch the `ModelicaSyntaxError`, it contains info pointing to the offending Modelica source code.

In [7]:
try:
    parser.parse_file(bad_src_file)
except parser.ModelicaSyntaxError as exc:
    print(type(exc).__name__)
    print("  file    :", exc.filename)
    print("  message :", exc.args[0])
    print("  line    :", exc.lineno)
    print("  col     :", exc.offset)
    print("  is SyntaxError:", isinstance(exc, SyntaxError))

ModelicaSyntaxError
  file    : /Users/rutanwk/Claude/tmp/pymoca/test/models/RedeclareNestedClass.mo.fail_parse
  message : missing '=' at '.'
  line    : 21
  col     : 48
  is SyntaxError: True


There is also a `print_syntax_error()` helper.

In [8]:
try:
    parser.parse_file(bad_src_file)
except parser.ModelicaSyntaxError as exc:
    parser.print_syntax_error(exc)

/Users/rutanwk/Claude/tmp/pymoca/test/models/RedeclareNestedClass.mo.fail_parse:21:48:
  extends D(c.z.x(nominal=2), redeclare model C.B=F);
ModelicaSyntaxError: missing '=' at '.'


**In pymoca 0.11.2:** `parse()` returned `None` for bad syntax. ANTLR printed an error directly to stderr with no way to intercept it.

In [9]:
run_old(f"""
from pymoca import parser
result = parser.parse(f'''{bad_src_file.read_text()}''')
print('result:', repr(result))
""")
# stderr contains the raw ANTLR token error; result is None


result: None
line 21:47 mismatched input '.' expecting '='


One final point to make about this example. All versions of pymoca accept *multiple
class definitions in one file*. This is a superset of the Modelica language spec. It
makes for simpler examples, but to be perfectly spec compliant a syntax error would the
raised for that error even if the syntax error that is shown above were fixed.

<a id="sec1-3"></a>
### 1.3 `tree.find_name(scope, name)` — look up a name in a class scope

`find_name()` resolves a Modelica name such as `A.B.C` given a starting scope of a
parsed or instantiated class, following the MLS §5 lookup rules.  Unlike the old
`find_class()` method on `ast.Class`, it returns **either** a class or a symbol
depending on what the name resolves to, and accepts a plain `str` in addition to
`ast.ComponentRef`.

In [10]:
# find_name returns ast.Class for a class name, ast.Symbol for a variable/constant
src_fn = '''
package Hydraulics
  constant Real rho = 1000.0;
  model Pipe
    Real p;
  end Pipe;
end Hydraulics;
'''
fn_tree = parser.parse(src_fn)
hydraulics = fn_tree.classes["Hydraulics"]

pipe_class = tree.find_name(hydraulics, "Pipe")
print("Pipe ->", type(pipe_class).__name__, ":", pipe_class.name)

rho_sym = tree.find_name(hydraulics, "rho")
print("rho  ->", type(rho_sym).__name__, ":", rho_sym.name)

Pipe -> Class : Pipe
rho  -> Symbol : rho


**In pymoca 0.11.2:** `ast.Class.find_class()` only searched the class
namespace for *classes* — symbols (constants, parameters, variables) could not
be found by name.  It also required an `ast.ComponentRef` argument rather than a
plain string.

In [11]:
run_old(f"""
import pymoca.parser as p, pymoca.ast as ast
src = '''{src_fn}'''
tree_obj = p.parse(src)
hydraulics = tree_obj.classes['Hydraulics']
pipe = hydraulics.find_class(ast.ComponentRef(name='Pipe'))
print('find_class Pipe:', type(pipe).__name__, ':', pipe.name)
rho = hydraulics.find_class(ast.ComponentRef(name='rho'))
print('find_class rho:', repr(rho), '<- cannot look up symbols')
""")


find_class Pipe: Class : Pipe
Traceback (most recent call last):
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/ast.py", line 644, in _find_class
    return self.classes[component_ref.name]
           ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
KeyError: 'rho'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/ast.py", line 686, in _find_class
    raise ClassNotFoundError
pymoca.ast.ClassNotFoundError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/ast.py", line 644, in _find_class
    return self.classes[component_ref.name]
           ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
KeyError: 'rho'

During handling of the above exception, another exception occurred:

Trac

<a id="sec1-4"></a>
### 1.4 `tree.NameLookupError`

The new pipeline raises `NameLookupError` when a dotted name cannot be resolved —
for example, when an intermediate segment of `A.B.C` is a model rather than a
package (MLS §5.3).

In pymoca 0.11.2, an unresolvable name returned `None`; the model flattened with
a wrong or absent value rather than raising an error.

In [12]:
src_nle = '''
model Counter
  Real count = 1.0;
end Counter;

model Bad
  Real x = Counter.count;  // Counter is a model, not a package
end Bad;
'''
t_nle = parser.parse(src_nle)
try:
    tree.flatten_class(t_nle, "Bad")
    print("No error raised")
except tree.NameLookupError as exc:
    print(f"NameLookupError: {exc}")

NameLookupError: Counter is not a package so count must be encapsulated


In [13]:
run_old(f"""
import pymoca.parser as p, pymoca.tree as t
tree_obj = p.parse('''{src_nle}''')
bad_class = tree_obj.classes['Bad']
flat = t.flatten_class(bad_class)
x_sym = flat.symbols.get('x')
print('x value:', repr(x_sym.value) if x_sym else 'symbol missing')
print('(no error raised — wrong value silently accepted)')
""")

x value: 'Counter.count'
(no error raised — wrong value silently accepted)


<a id="sec1-5"></a>
### 1.5 `tree.instantiate(ast_tree, name)` — next step after parse

`instantiate()` resolves the class hierarchy, applies modifications, and builds the
instance tree — but stops before equation flattening.  The result is an `InstanceClass`
with `parent`, `parent_instance`, and `ast_ref` links.  This composable step is useful
for tooling (e.g. a type checker or documentation generator) that needs the resolved
class structure without needing flattened equations. 

To demonstrate, we'll return to the gear model from above.

In [14]:
# New: instantiate without flattening
instance = tree.instantiate(ast_tree, "GearType1")
print(type(instance).__name__)    # InstanceClass
print("name :", instance.name)
print("parent :", type(instance.parent).__name__)
print("ast_ref:", instance.ast_ref)
print("symbols:", list(instance.symbols))

InstanceClass
name : GearType1
parent : InstanceTree
ast_ref: Class GearType1, Type "model"
symbols: ['bearingFriction']


**In pymoca 0.11.2:** `tree.build_instance_tree()` is the comparable function that
resolves extends clauses and shifts modifications, but is missing some steps from the
Modelica Language Spec that may cause it to fail for more complicated models.

In [15]:
run_old(f"""
from pymoca import ast, parser, tree
ast_tree = parser.parse('''{modelica_file.read_text()}''')
instance = tree.build_instance_tree(ast_tree.classes["GearType1"])
""")  # Name lookup error due to scoping


Traceback (most recent call last):
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/ast.py", line 647, in _find_class
    return self.classes[component_ref.name]._find_class(component_ref.child[0], False)
           ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
KeyError: 'Modelica'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/ast.py", line 686, in _find_class
    raise ClassNotFoundError
pymoca.ast.ClassNotFoundError

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "<string>", line 37, in <module>
    instance = tree.build_instance_tree(ast_tree.classes["GearType1"])
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/tree.py", line 446, in build_instance_tree
    c = extended_orig_class.find_class(s

<a id="sec1-6"></a>
### 1.6 `tree.flatten_instance(instance)` —  generate flat class from instance

`flatten_instance()` takes an instance produced by `instantiate()` and returns a
flat `InstanceClass` with dotted symbol names, resolved equations, and expanded
connect clauses.  The result carries `parent`, `parent_instance`, and `ast_ref`
links to the parsed and instance trees, which backends and debugging tools can use.

We'll use a water channel hydrological model from the pymoca test suite that pymoca 0.11.2 can flatten.

In [16]:
modelica_file = MODEL_DIR / "ExtendsModification.mo"
print(modelica_file.read_text())

connector HQPort
  Real H;
  flow Real Q;
end HQPort;

partial model HQOnePort
  HQPort HQ;
end HQOnePort;

partial class Volume
  Real V(min = 0, nominal = 1e6);
end Volume;

partial model PartialStorage
  extends HQOnePort;
  extends Volume;
equation
  der(V) = HQ.Q;
end PartialStorage;

model Linear
  extends PartialStorage(HQ.H(min = H_b));
  parameter Real A;
  parameter Real H_b;
equation
  V = A * (HQ.H - H_b);
end Linear;

model MainModel
	Linear e(H_b=-2.0, A=1000.0);
equation
end MainModel;

// LinearExpr: same as Linear but the cross-scope modification value is an
// Expression (2 * H_b) rather than a bare ComponentRef.  This exercises the
// expression-operand path of _resolve_modification_attribute with a
// base-class scope (PartialStorage) that differs from the flat-naming root
// (MainModelExpr).
model LinearExpr
  extends PartialStorage(HQ.H(min = 2 * H_b));
  parameter Real A;
  parameter Real H_b;
equation
  V = A * (HQ.H - H_b);
end LinearExpr;

model MainModelExpr


In [17]:
# New: flatten_instance → InstanceClass
ast_tree = parser.parse_file(modelica_file)
instance = tree.instantiate(ast_tree, "MainModel")
flat_instance = tree.flatten_instance(instance)
print("type:", type(flat_instance).__name__)        # InstanceClass
print("symbols:", sorted(flat_instance.symbols))  # Includes inherited symbols, now flattened
# Symbol values are plain Python values (not Primary-wrapped)
symbol = flat_instance.symbols["e.A"]
print("e.A value:", symbol.value, "of type", type(symbol.value).__name__)
print("e.A type:", type(symbol).__name__)

type: InstanceClass
symbols: ['e.A', 'e.HQ.H', 'e.HQ.Q', 'e.H_b', 'e.V']
e.A value: 1000.0 of type float
e.A type: InstanceSymbol


**In pymoca 0.11.2:** the comparable function is `tree.flatten_class()`, which takes a
parsed `ast.Class` (looked up from the tree by name) and returns a flat `ast.Class`.
Symbol values are `Primary`-wrapped, so accessing a float value requires `.value.value`
rather than `.value`.

In [18]:
run_old(f"""
from pymoca import parser, tree, ast
src = open(r'{modelica_file}').read()
ast_tree = parser.parse(src)
model_class = ast_tree.classes['MainModel']
flat_class  = tree.flatten_class(model_class)
print('type:', type(flat_class).__name__)
print('symbols:', sorted(flat_class.symbols))  # Extra symbol e.HQ
e_A = flat_class.symbols['e.A']
print('e.A value:', repr(e_A.value))  # Primary(value=1000.0)
print('e.A value.value:', e_A.value.value, 'of type', type(e_A.value.value).__name__)
print()
print('equations (first 2):')
[print(' ', repr(eq)) for eq in flat_class.equations[:2]]
""")


type: Class
symbols: ['e.A', 'e.HQ', 'e.HQ.H', 'e.HQ.Q', 'e.H_b', 'e.V']
e.A value: Primary(value=1000.0)
e.A value.value: 1000.0 of type float

equations (first 2):
  Equation(left=Expression(operator='der', operands=['e.V']), right='e.HQ.Q')
  Equation(left='e.V', right=Expression(operator='*', operands=['e.A', Expression(operator='-', operands=['e.HQ.H', 'e.H_b'])]))


We have an extra symbol `e.HQ` in the pymoca 0.11.2 flattened class that does not show up in the new code output or when flattened by other Modelica tools.

<a id="sec1-7"></a>
### 1.7 `tree.flatten_class(root, class_name)` — flatten to `InstanceClass`

Pymoca 0.11.2 has a `flatten_class()` with a different signature (see above). The
function name has been recycled in the new flattening as `flatten_class(root,
class_name)`: a single call that takes a parsed tree and a class name, and returns a
flat class result.

Optional flags on `flatten_class()`:

- `keep_connectors=True` — preserve connector symbols and unexpanded `ConnectClause`
  objects. This can be useful to pre-flatten connectable components.
- `evaluate_parameters=True` — fold parameter values into the equations numerically.

In [19]:
# New flatten_class() signature — equivalent to flatten_instance(instantiate(tree, name))
mm_flat = tree.flatten_class(ast_tree, "MainModel")
print("symbols:", sorted(mm_flat.symbols))
print("e.A value:", mm_flat.symbols["e.A"].value, "—", type(mm_flat.symbols["e.A"].value).__name__)

# evaluate_parameters=True folds parameter cross-references to numeric Primary values
src_ep = """
model DerivedParam
  parameter Real q = 5.0;
  parameter Real p = q;
  Real x = p;
end DerivedParam;
"""
ep_tree = parser.parse(src_ep)

flat_default = tree.flatten_class(ep_tree, "DerivedParam")
print("\nWithout evaluate_parameters:")
print("  p.value:", flat_default.symbols["p"].value)   # InstanceSymbol reference to q

flat_ep = tree.flatten_class(ep_tree, "DerivedParam", evaluate_parameters=True)
print("With evaluate_parameters=True:")
print("  p.value:", flat_ep.symbols["p"].value)        # Primary(value=5.0) — folded


symbols: ['e.A', 'e.HQ.H', 'e.HQ.Q', 'e.H_b', 'e.V']
e.A value: 1000.0 — float

Without evaluate_parameters:
  p.value: InstanceSymbol q, Type "InstanceClass Real, Type "type""
With evaluate_parameters=True:
  p.value: Primary value 5.0


<a id="sec1-8"></a>
### 1.8 `tree.flatten(ast_tree, name)` — flatten to `ast.Tree`

`flatten()` runs the new pipeline like `flatten_instance()` but additionally converts
the `InstanceClass` result back to a plain `ast.Tree` containing `ast.Class` and
`ast.Symbol` objects so it is is backward-compatible with existing CasADi / SymPy
backends and they continue to work without modification.

Both new and 0.11.2 go through multiple internal passes. The new
pipeline is more MLS-compliant at each pass (proper modification scoping, dual-tree
pointers, redeclare-as-extends), producing the same final `ast.Tree` shape. Note that `flatten()` accepts a `str` to identify the class to be flattened. It also accepts `ast.ComponentRef` as it does in version 0.11.2.

In [20]:
# tree.flatten returns an ast.Tree in the form current backends expect
flat_tree = tree.flatten(ast_tree, "MainModel")
print("result type:", type(flat_tree).__name__)
mm_cls = flat_tree.classes["MainModel"]
print("class type :", type(mm_cls).__name__)
print("symbols    :", sorted(mm_cls.symbols))
print("e.A value  :", mm_cls.symbols["e.A"].value)   # Primary-wrapped (legacy format)

# The CasADi backend calls tree.flatten() internally — it still works unchanged
from pymoca.backends.casadi.api import transfer_model
casadi_model = transfer_model(str(MODEL_DIR), "MainModel")
print("\nCasADi states    :", casadi_model.states)
print("CasADi parameters:", casadi_model.parameters)
print("CasADi equations :", casadi_model.equations)


result type: Tree
class type : Class
symbols    : ['e.A', 'e.HQ.H', 'e.HQ.Q', 'e.H_b', 'e.V']
e.A value  : Primary value 1000.0



CasADi states    : [e.V[1,1]:float]
CasADi parameters: [e.A[1,1]:float, e.H_b[1,1]:float]
CasADi equations : [MX((der(e.V)-e.HQ.Q)), MX((e.V-(e.A*(e.HQ.H-e.H_b)))), MX(e.HQ.Q)]


**In pymoca 0.11.2:** This model from the 0.11.2 test suite gives the same results. Note that `flatten()` in that version requires an `ast.ComponentRef()`.

In [21]:
run_old(f"""
from pymoca import parser, tree, ast
from pymoca.backends.casadi import generator
with open(r'{modelica_file}') as f:
    src = f.read()

# Old flatten() requires ComponentRef, not a plain string
ast_tree = parser.parse(src)
flat_tree = tree.flatten(ast_tree, ast.ComponentRef(name='MainModel'))
mm_cls = flat_tree.classes['MainModel']
print('result type:', type(flat_tree).__name__)
print('symbols    :', sorted(mm_cls.symbols))
print('e.A value  :', repr(mm_cls.symbols['e.A'].value))  # Primary-wrapped

# Parse again — old tree.flatten() mutates ast_tree in place, so generator
# needs a fresh parse to avoid a corrupted tree
ast_tree2 = parser.parse(src)
casadi_model = generator.generate(ast_tree2, 'MainModel')
print()
print('CasADi states    :', casadi_model.states)
print('CasADi parameters:', casadi_model.parameters)
print('CasADi equations :', casadi_model.equations)
""")


result type: Tree
symbols    : ['e.A', 'e.HQ.H', 'e.HQ.Q', 'e.H_b', 'e.V']
e.A value  : Primary(value=1000.0)

CasADi states    : [e.V[1,1]:float]
CasADi parameters: [e.A[1,1]:float, e.H_b[1,1]:float]
CasADi equations : [MX((der(e.V)-e.HQ.Q)), MX((e.V-(e.A*(e.HQ.H-e.H_b)))), MX(e.HQ.Q)]


<a id="sec1-9"></a>
### 1.9 `tree.ModelicaSemanticError`

The new pipeline raises `ModelicaSemanticError` for programs that parse correctly
but violate a semantic rule: self-extends, extending a `replaceable` class, or
redeclaring a non-`replaceable` element (MLS §7.1.3, §7.3).

In pymoca 0.11.2, self-extends raised a generic error; other violations were
silently accepted, producing wrong or incomplete output.

See [Part 4](#part4) for detailed examples.

<a id="part2"></a>
## Part 2 — MODELICAPATH (MLS §13.2.3)

pymoca now supports the `MODELICAPATH` mechanism: a list of directories where
package roots are discovered automatically.  `parser.modelicapath_to_tree()` takes
a list of such directories and returns a **lazy-parse stub tree** — the package
hierarchy is navigable immediately, but individual files are only parsed on demand
when a class is actually needed for flattening.

This is what enables testing pymoca against the full Modelica Standard Library (MSL)
without pre-parsing all ~2000 `.mo` files up-front.

In [22]:
# Build a lazy stub tree from the MSL 4.0.x directory
msl_tree = parser.modelicapath_to_tree([str(MSL4_DIR)])
print(type(msl_tree).__name__)

# The top-level package is immediately visible
msl_pkg = msl_tree.classes.get("Modelica")
print("Top-level class:", type(msl_pkg).__name__, "name:", msl_pkg.name if msl_pkg else "(not found)")

# Navigate into a sub-package using find_name — triggers lazy parsing of only what's needed
si = tree.find_name(msl_tree, "Modelica.Units.SI")
print("Modelica.Units.SI:", type(si).__name__)

Tree
Top-level class: LazyParseClass name: Modelica


Modelica.Units.SI: Class


In [23]:
# Flatten a real MSL model using the MODELICAPATH tree
msl_bb_flat = tree.flatten_class(msl_tree, "Modelica.Mechanics.Rotational.Examples.First")
print("Flattened:", msl_bb_flat.name)
print("Symbols   :", list(msl_bb_flat.symbols)[:6], "...")
print("Equations :", len(msl_bb_flat.equations))

Flattened: First
Symbols   : ['amplitude', 'f', 'Jmotor', 'Jload', 'ratio', 'damping'] ...
Equations : 58


**In pymoca 0.11.2:** there was no MODELICAPATH support.  Every file had to be
passed explicitly to `parse()`, and the user was responsible for locating and
listing all transitive dependencies.

<a id="part3"></a>
## Part 3 — The `pymoca` CLI (`compiler.py`)

`src/pymoca/compiler.py` is now a proper package entry point: `pip install pymoca`
puts `pymoca` on `PATH`.  The CLI exposes the full pipeline with granular control:

```
pymoca [PATHNAME ...] [-p PATH] [-m MODEL]
       [--stage {parse,instantiate,flatten,translate}]
       [--format {json,repr}]
       [-t {casadi,sympy}]
```

The `--stage` flag stops the pipeline at any step and prints the result to stdout,
making it a useful debugging and scripting tool.  For example,
`pymoca -p MSL-4.0.x -m Modelica.Blocks.Continuous.PI --stage flatten --format repr | head`
gives you the flat instance without writing any Python.

In [24]:
# Run the new CLI to flatten a simple MSL model and show its flat repr
result = subprocess.run(
    [sys.executable, "-m", "pymoca.compiler",
     "-p", str(MSL4_DIR),
     "-m", "Modelica.Mechanics.Rotational.Examples.First",
     "--stage", "flatten",
     "--format", "repr"],
    capture_output=True,
    text=True,
    env={**os.environ},   # new CLI uses in-process pymoca, keep PYTHONPATH
    timeout=60,
)
print(result.stdout[:800] if result.stdout else "(no output)")
if result.returncode != 0:
    print("STDERR:", result.stderr[:400])

InstanceClass(name='First', ast_ref=Class(name='First', type='model'), modification_environment=ClassModification(arguments=[]))



**In pymoca 0.11.2:** `compiler.py` is a development script located in the `tools/`
directory of the source checkout.  It is **not** installed with `pip install pymoca`,
so users must locate and run it manually.  The script accepts explicit file
paths only (no MODELICAPATH support), so using MSL classes requires listing every
dependency file by hand.

<a id="part4"></a>
## Part 4 — Modelica Language Specification compliance

This section covers behaviors where the **result** produced by new pymoca differs
from pymoca 0.11.2.

### 4.1 AST representation after instantiation

The new code maintains a **dual-tree model**: every node carries both a *lexical*
(`parent`) reference and an *instance* (`parent_instance`) referece.

- **`parent`** — the lexical scope in which the element was declared.  Name lookup
  climbs the lexical chain per MLS §5.3 (up through enclosing classes, across
  `extends` clauses at the declaration site).
- **`parent_instance`** — the instance-tree parent (the class being built).
  Flat-name generation and modification application follow this chain, because the
  flat namespace corresponds to the *instance* hierarchy, not the lexical one.

Without both, modification scoping (e.g. in `extends Base(x = y)` resolving `y` against
the model being instantiated and later flattened) is difficult to implement correctly.
These two references can point to the same python object, but often are different (MLS
§5.6.2).

#### Unnamed `extends` nodes

When a class `extends Base`, the instance tree inserts an anonymous `InstanceClass` node
representing the base class content into the derived class (MLS §5.6.1.4).  These
unnamed nodes carry the base class content but have `parent_instance` pointing to the
derived class, preserving the instance path for flat-naming and retain `parent` for
correct lexical scoping. Pymoca 0.11.2 did the equivalent of `parent_instance` by
immediately copying the `extends Base` contents into the derived class, but this made
later lexical resolution difficult, leading to issues flattening some models that
prompted the new implementation.

#### `redeclare` as `extends`

A `redeclare` modifier is represented internally as an `extends` node in the instance
tree (MLS §5.6.1.4).  This means a class can accumulate multiple extends entries — its
explicitly written extends clauses plus one entry per `redeclare` applied to it — and
the same two-pass extends algorithm handles both.

In [25]:
# Show parent / parent_instance with extends and redeclare (MLS §5.6.1.4)
src_dt = """
package Pkg
  model Base
    replaceable type T = Real(max=100.0);
  end Base;
  model Derived
    type MyType = Real(max=50.0);
    extends Base(redeclare type T = MyType);
    parameter T extra = 42.0;
  end Derived;
end Pkg;
"""
dt_tree = parser.parse(src_dt)
print("classes in Pkg:", list(dt_tree.classes["Pkg"].classes))


classes in Pkg: ['Base', 'Derived']


We call `tree.instantiate()` on `Pkg.Derived` and inspect the resulting `InstanceClass` to verify the dual-tree structure described above.  `parent` tracks the lexical scope (`Pkg`), `parent_instance` tracks the instance hierarchy, the `Base` extends clause becomes an anonymous `InstanceClass` node (name=`''`) whose `parent_instance` is `Derived`, and `extra` is an `InstanceSymbol` — not a plain `ast.Symbol`.

In [26]:
# Pkg.Derived extends Base and carries a class redeclare in the extends clause
derived_inst = tree.instantiate(dt_tree, "Pkg.Derived")
print("InstanceClass  :", type(derived_inst).__name__)
print("ast_ref        :", derived_inst.ast_ref)
print("parent         :", derived_inst.parent)           # lexical parent: Pkg package
print("parent_instance:", derived_inst.parent_instance)  # instance parent: root tree
print()

# .extends: one unnamed InstanceClass per extends clause (redeclares handled inside)
print(".extends count:", len(derived_inst.extends))
ext = derived_inst.extends[0]     # the Base extends clause
print("  .extends[0].name           :", repr(ext.name))   # "" — unnamed
print("  .extends[0].ast_ref        :", ext.ast_ref)
print("  .extends[0].parent         :", ext.parent)        # lexical: Pkg
print("  .extends[0].parent_instance:", ext.parent_instance)  # instance: Derived
print()

# Symbols directly declared in Derived (inherited symbols live in the extends node)
print("symbols:", list(derived_inst.symbols))
extra = derived_inst.symbols["extra"]
print("extra:", type(extra).__name__, "— parent_instance:", extra.parent_instance.name)


InstanceClass  : InstanceClass
ast_ref        : Class Derived, Type "model"
parent         : InstanceClass Pkg, Type "package"
parent_instance: InstanceTree None, Type "package"

.extends count: 1
  .extends[0].name           : ''
  .extends[0].ast_ref        : Class Base, Type "model"
  .extends[0].parent         : InstanceClass Pkg, Type "package"
  .extends[0].parent_instance: InstanceClass Derived, Type "model"

symbols: ['extra']
extra: InstanceSymbol — parent_instance: Derived


Instantiation in Modelica is all about resolving components/variables, represented as `ast.Symbol` in pymoca. By focusing variables, we don't waste time processing all the other things that are not important for a given model. At this stage the object-oriented, hierarchical relationships are still represented in the AST. The extends and redeclare information we expect to see in this example are contained in the type of the symbol `extra`.

In [27]:
# extra.type is an InstanceClass — OO hierarchy preserved before flattening
T_inst = extra.type
print('extra.type:', type(T_inst).__name__, T_inst.name)
print('  .ast_ref (declared in):', T_inst.ast_ref)   # Base's T
print('  .parent (lexical scope):', T_inst.parent)    # InstanceClass Base
print('  .parent_instance (instance scope):', T_inst.parent_instance)    # InstanceClass T

# Redeclare (T = MyType) shows up as an unnamed extends node inside T's instance
print('  .extends count:', len(T_inst.extends))
print('  .extends[0].ast_ref:', T_inst.extends[0].ast_ref)  # MyType — the redeclared type

extra.type: InstanceClass T
  .ast_ref (declared in): Class T, Type "type"
  .parent (lexical scope): InstanceClass Base, Type "model"
  .parent_instance (instance scope): InstanceSymbol extra, Type "InstanceClass T, Type "type""
  .extends count: 1
  .extends[0].ast_ref: Class MyType, Type "type"


**In pymoca 0.11.2:** `InstanceClass` has no `parent_instance` and no `ast_ref`.
There is no `InstanceSymbol` type — symbols are plain `ast.Symbol` objects. The output from `build_instance_tree()` has semi-flattened `extends` by copying the contents into the derived class but names are not flattened at this point. Redeclares are similarly semi-flattened. The function `build_instance_tree()` is an internal step on the way to flattening, but the implementation differs at this point from the process described in the spec.

In [28]:
run_old(f"""
import pymoca.parser as p, pymoca.tree as t
src = '''{src_dt}'''
tree_obj = p.parse(src)
cls_D = tree_obj.classes['Pkg'].classes['Derived']
inst = t.build_instance_tree(cls_D)
print('type             :', type(inst).__name__)
print('has parent_instance:', hasattr(inst, 'parent_instance'))
print('has ast_ref        :', hasattr(inst, 'ast_ref'))
sym = inst.symbols.get('extra')
print('symbol type        :', type(sym).__name__ if sym else '(none)')
# Semi-flattened: Base's T was copied into Derived's classes; extends is empty
print('extends            :', inst.extends)
print('classes            :', list(inst.classes.keys()))
# OO hierarchy collapsed: extra.type is already MyType, not T
print('extra.type         :', sym.type)
""")

type             : InstanceClass
has parent_instance: False
has ast_ref        : False
symbol type        : Symbol
extends            : []
classes            : ['T', 'MyType']
extra.type         : InstanceClass MyType, Type "__builtin"


### 4.2 AST representation after flattening

After flattening, the new `flatten_instance()` and `flatten_class()` produce a flat `InstanceClass` — the same type as after instantiation, but with:
- **dotted symbol names** (`a.up.H` instead of nested objects)
- **equations collected** from all extends chains into a single list
- **value modifications converted to equations** for variables
  (e.g. `Real x = y + 1` becomes an equation; `parameter Real p = 3.0` keeps its
  value on the symbol)
- **connect clauses expanded** to equality + zero-sum flow equations

The new `flatten()` masks the value-modification → equation conversion for legacy
backends that read symbol values directly and it produces an `ast.Tree` that the backends expect.

In 0.11.2, `tree.flatten()` returns a flat `ast.Tree` (not `InstanceClass`), with
all symbol values retained as `Primary`-wrapped objects and no `parent_instance`
or `ast_ref`.  See §1.4 for the side-by-side comparison.

### 4.3 Built-in type names are reserved (MLS §4.8)

`Real`, `Integer`, `Boolean`, `String` are placed at the root of the class tree
by the parser.  Declaring `model Real ... end Real;` is now a parse error.

In [29]:
try:
    src = """
    package P
      model Real Real x = 1.0; end Real;
      model UseReal Real y; end UseReal;
    end P;
    """
    parser.parse(src)
except parser.ModelicaSyntaxError as exc:
    print(type(exc).__name__, ":", exc)

ModelicaSyntaxError : Predefined type Real not allowed as identifier (, line 3)


**In pymoca 0.11.2:** `model Real` is accepted but is bypassed by the special-cases used in the the pymoca code for the built-in types and name lookup of the simple built-in types throws an exception.

In [30]:
run_old(f"""
from pymoca import parser, ast
result = parser.parse('''{src}''')
use_real = result.classes['P'].classes['UseReal']
use_real_y = use_real.symbols['y']
print('UseReal.y:', repr(use_real_y))
print(use_real.find_class(ast.ComponentRef(name='Real')))
""")


UseReal.y: Symbol(name='y', type='Real')
Traceback (most recent call last):
  File "<string>", line 12, in <module>
    print(use_real.find_class(ast.ComponentRef(name='Real')))
          ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/ast.py", line 716, in find_class
    raise FoundElementaryClassError()
pymoca.ast.FoundElementaryClassError


### 4.4 Modification scoping (MLS §4.5.1, §5.6)

`ModificationScopeFlatten.mo` is derived from an example in MLS §4.5.1.  Model `A`
defines `model Load = Resistor(R=R)` — the RHS `R` must resolve in `A`'s enclosing scope
(where `R = 3.0`), **not** inside `Resistor`.  When model `B` instantiates `Load b`,
flattening should yield `b.R` referencing `B`'s `R` (= 3.0).  In 0.11.2 the modification
is resolved in the wrong scope with a value of `1.0`.

In [31]:
scope_file = MODEL_DIR / "ModificationScopeFlatten.mo"
scope_tree = parser.parse(scope_file.read_text())
flat_class = tree.flatten_class(scope_tree, "B", evaluate_parameters=True)
print("b.R value:", repr(flat_class.symbols["b.R"].value))

b.R value: Primary(value=3)


In [32]:
run_old(f"""
from pymoca import parser, tree, ast
with open('{scope_file}') as fh:
    src = fh.read()
tree_obj = parser.parse(src)
flat_tree = tree.flatten(tree_obj, ast.ComponentRef(name='B'))
flat = flat_tree.classes.get('B') or list(flat_tree.classes.values())[-1]
b_R = flat.symbols['b.R'].value
print('b.R value:', repr(b_R))
""")

b.R value: Primary(value=1)


### 4.5 Expression evaluation in modifications (MLS §4.5.1, §5.6 step B)

Modifications can contain literal values (`a = 3.0`), references to other
parameters (`b = a`), or expressions (`c = 2.0 * 3.0 + 1.0`).  The new code
unwraps literal `Primary` scalars to plain Python values at flatten time.  Flat
single-level expressions (e.g. unary negation `-3.0`) are also evaluated.  Nested
or parameter-dependent expressions are kept as `Expression` AST nodes in both old
and new code — neither performs deep constant folding.

Three spec rules drive when evaluation must happen at flatten time:

- **§4.5.1 (short class definitions)** — `class A = B(x = expr)` is the sharpest case:
  `expr` must be resolved in A's *enclosing* scope, not inside B.  The new code tracks
  this via `ClassModificationArgument.scope` so the right scope is used when folding.
- **§5.6.2 (conditional declarations)** — `if`-guarded components must be evaluated to
  determine which components exist; this is a hard spec requirement.
- **Appendix B (DAE)** — parameters and constants are static in the DAE representation,
  so their values must be resolved before the flat model is emitted.  Array dimensions
  (`Real x[n]`) fall here too: `n` must be parameter-variability and foldable to a
  concrete integer (§10.1).

In [33]:
src_expr = '''
package P
  model A
    parameter Real a = 3.0;             // literal Primary
    parameter Real b = a;               // ComponentRef to another parameter
    parameter Real c = 2.0 * 3.0 + 1.0; // nested expression
    Real x = a * b + 1.0;               // non-parameter: becomes equation
  end A;
  model B
    A d;
  end B;
end P;
'''
tree_obj = parser.parse(src_expr)
flat_class = tree.flatten_class(tree_obj, "P.B")
print("d.a (literal)    :", repr(flat_class.symbols["d.a"].value))
print("d.b (= d.a, ref) :", repr(flat_class.symbols["d.b"].value))
print("d.c (nested expr):", repr(flat_class.symbols["d.c"].value))
print("equations        :", flat_class.equations)

d.a (literal)    : 3.0
d.b (= d.a, ref) : InstanceSymbol(name='d.a', ast_ref=Symbol(name='a', type='Real'), modification_environment=ClassModification(arguments=[]))
d.c (nested expr): Expression(operator='+', operands=[Expression(operator='*', operands=[Primary(value=2.0), Primary(value=3.0)]), Primary(value=1.0)])
equations        : [Equation(left='d.x', right=Expression(operator='+', operands=[Expression(operator='*', operands=['d.a', 'd.b']), Primary(value=1.0)]))]


**In pymoca 0.11.2:** literal parameter values are `Primary`-wrapped; `ComponentRef`
modifications (`b = a`) are reduced to the flat name string; nested expression
modifications are stored unevaluated — the same as new code.

In [34]:
run_old(f"""
from pymoca import parser, tree, ast
src = '''{src_expr}'''
tree_obj = parser.parse(src)
flat_tree = tree.flatten(tree_obj, ast.ComponentRef.from_string("P.B"))
flat_class = flat_tree.classes['P.B']
print('d.a (literal)    :', repr(flat_class.symbols['d.a'].value))
print('d.b (= d.a, ref) :', repr(flat_class.symbols['d.b'].value))
print('d.c (nested expr):', repr(flat_class.symbols['d.c'].value))
print('equations        :', flat_class.equations)
""")

d.a (literal)    : Primary(value=3.0)
d.b (= d.a, ref) : 'd.a'
d.c (nested expr): Expression(operator='+', operands=[Expression(operator='*', operands=[Primary(value=2.0), Primary(value=3.0)]), Primary(value=1.0)])
equations        : [Equation(left=Symbol(name='d.x', type='Real'), right=Expression(operator='+', operands=[Expression(operator='*', operands=['d.a', 'd.b']), Primary(value=1.0)]))]


### 4.6 Component redeclare in extends (MLS §7.3)

The new flatten pipeline fully supports replacing a `replaceable` component’s type and
value inside an `extends` modifier. In 0.11.2, component redeclare was only partially
functional: the type change would take effect, but modifications on built-in-typed
components (`Real`, `Integer`, etc.) were silently discarded.  `redeclare Real x = 4.0`
now correctly preserves the `= 4.0` modification.

In [35]:
src_rd = '''
model Base
  replaceable parameter Real x = 1.0;
end Base;

model RedeclareBuiltin
  extends Base(redeclare parameter Real x = 4.0);
end RedeclareBuiltin;
'''
t5 = parser.parse(src_rd)
flat5 = tree.flatten_class(t5, "RedeclareBuiltin")
print("x value :", flat5.symbols["x"].value)   # should be 4.0
assert flat5.symbols["x"].value == 4.0
print("Modification preserved ✓")

x value : 4.0
Modification preserved ✓


**In pymoca 0.11.2:** built-in types (`Real`, `Integer`, `Boolean`, `String`) are
re-instantiated from scratch on each component redeclare, so the `= 4.0`
modification is silently dropped.  Modifications on non-built-in component
redeclares (e.g. `redeclare MyClass comp(param=5.0)`) propagate correctly in
both versions through normal sub-component modification mechanics.

In [36]:
run_old(f"""
from pymoca import parser, tree, ast
src = '''{src_rd}'''
tree_obj = parser.parse(src)
result = tree.flatten(tree_obj, ast.ComponentRef(name='RedeclareBuiltin'))
cls = result.classes['RedeclareBuiltin']
x = cls.symbols['x']
print('x value:', repr(x.value))  # Primary(value=1.0) — modification lost
""")


x value: Primary(value=1.0)


### 4.7 User-defined enumerations (MLS §4.8.5)

Enum short-class definitions parse and flatten without error — enough to let MSL
examples through.  Each literal carries a 1-based ordinal in the parsed AST.
`Integer()` conversion and comparison operators are not evaluated during flattening;
they pass through as unevaluated expressions.

In [37]:
src_enum = '''
type Color = enumeration(red, green, blue);
model UseEnum
  Color c = Color.green;
  parameter Integer n = Integer(Color.green);
end UseEnum;
'''
ast_tree = parser.parse(src_enum)

# Literals in the parsed class carry 1-based ordinals
print("Color literals with ordinals:")
for name, lit in ast_tree.classes["Color"].symbols.items():
    print(f"  Color.{name}: ordinal={lit.ordinal}")

# Enum-typed symbols flatten without error
flat_class = tree.flatten_class(ast_tree, "UseEnum", evaluate_parameters=True)
c_sym = flat_class.symbols["c"]

# c value is Primary(value=None) -- the initializer stays as an equation, not resolved
print(f"\nc.type.name: {repr(c_sym.type.name)}  c.value: {repr(c_sym.value)}")
print(f"equation rhs: {repr(flat_class.equations[0].right)}")

# Integer() passes through as an unevaluated Expression
n_sym = flat_class.symbols["n"]
print(f"\nn.value: {repr(n_sym.value)}  (not evaluated to 2)")

Color literals with ordinals:
  Color.red: ordinal=1
  Color.green: ordinal=2
  Color.blue: ordinal=3

c.type.name: 'Color'  c.value: Primary(value=None)
equation rhs: 'Color.green'

n.value: Expression(operator='Integer', operands=['Color.green'])  (not evaluated to 2)


**In pymoca 0.11.2:** enumerations are not supported.

### 4.8 Leading-dot global names (MLS §B.2.7, §5.3.3)

The grammar now accepts leading-dot (globally-rooted) names such as `.Resistor` or
`.Modelica.Units.SI.Voltage` without a parse error.  The semantic side — resolving the
leading dot to global scope (MLS §5.3.3) — is not yet implemented, so models that
actually *use* such names still fail at the flattening stage.

In [38]:
src_ldot = '''
model Resistor parameter Real r = 1.0; end Resistor;
encapsulated model Load
  .Resistor r(r=2.0);
end Load;
'''
# Grammar now accepts leading-dot syntax; no parse error
ast_tree = parser.parse(src_ldot)
print("Parsed classes:", list(ast_tree.classes))

# Semantic resolution is not yet implemented; flattening raises
try:
    flat_class = tree.flatten_class(ast_tree, "Load", evaluate_parameters=True)
    print("Flat symbols:", list(flat_class.symbols))
except tree.NameLookupError as e:
    print(f"NameLookupError (expected): {e}")

Parsed classes: ['Real', 'Integer', 'Boolean', 'String', 'StateSelect', 'Clock', 'Resistor', 'Load']
NameLookupError (expected): Type Resistor of symbol r not found in Load


**In pymoca 0.11.2:** leading-dot syntax caused a grammar-level parse error.

### 4.9 `while` / `break` / `return` / `initial` now parsed (MLS §11)

These statement forms are rejected or silently dropped by pymoca 0.11.2.
They are now parsed correctly (though not yet semantically evaluated during flattening).

In [39]:
src_while = '''
function countdown
  input Integer n;
  output Integer result;
algorithm
  result := 0;
  while n > 0 loop
    result := result + n;
    n := n - 1;
  end while;
end countdown;
model UseWhile
  Integer r = countdown(3);
end UseWhile;
'''
# New: parses without error
t10 = parser.parse(src_while)
print("Parsed classes:", list(t10.classes))

Parsed classes: ['Real', 'Integer', 'Boolean', 'String', 'StateSelect', 'Clock', 'countdown', 'UseWhile']


**In pymoca 0.11.2:** `while` loops cause a parse error or are silently dropped.

In [40]:
run_old("""
import pymoca.parser as p
src = '''function countdown
  input Integer n;
  output Integer result;
algorithm
  result := 0;
  while n > 0 loop
    result := result + n;
    n := n - 1;
  end while;
end countdown;
model UseWhile
  Integer r = countdown(3);
end UseWhile;'''
result = p.parse(src)
print('result:', repr(result)[:120] if result else 'None (parse failed)')
""")

Traceback (most recent call last):
  File "<string>", line 16, in <module>
    result = p.parse(src)
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/parser.py", line 1095, in parse
    tree = _parse(txt)
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/pymoca/parser.py", line 827, in _parse
    parse_walker.walk(ast_listener, parse_tree)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/antlr4/tree/Tree.py", line 160, in walk
    self.walk(listener, child)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/antlr4/tree/Tree.py", line 160, in walk
    self.walk(listener, child)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/Users/rutanwk/Claude/tmp/pymoca/.venv-pymoca-011/lib/python3.14/site-packages/antlr4/tree/Tree.py", line 160, in walk
    self.walk(listener,

### 4.10 Name-lookup and semantic errors

The new code raises specific, informative exceptions for two classes of errors that
pymoca 0.11.2 either silently ignores, raises a generic Python error, or does not handle, resulting in a Python stack trace.

### 4.11 `tree.NameLookupError` — non-package used as a namespace

MLS §5.3: only a *package* class may be used to qualify a component name with dot
notation (`P.x`).  If `P` is a regular model or block, `P.x` is not allowed.

In [41]:
# New: NameLookupError when a non-package class is used as a namespace
src = '''
package P
  model Inner
    Real x = 1.0;
  end Inner;
  model BadLookup
    Inner inner_inst;
    Real y = Inner.x;   // Inner is not a package — MLS 5.3 violation
  end BadLookup;
end P;
'''
try:
    ast_tree = parser.parse(src)
    instance = tree.instantiate(ast_tree, "P.BadLookup")
    tree.flatten_instance(instance)
except tree.NameLookupError as exc:
    print(type(exc).__name__, ":", exc)

NameLookupError : Inner is not a package so x must be encapsulated


**In pymoca 0.11.2:** The appearance of `Inner.x` in the symbols of `BadLookup` indicates that the the lookup of `Inner.x` succeeds incorrectly according to the spec.

In [42]:
run_old(f"""
from pymoca import parser, tree, ast
src = '''{src}''' # From above
tree_obj = parser.parse(src)
result = tree.flatten(tree_obj, ast.ComponentRef.from_string('P.BadLookup'))
bad_lookup = result.classes.get('P.BadLookup', result.classes.get('BadLookup'))
bad_lookup_symbols  = list(bad_lookup.symbols.keys())
print('BadLookup symbols:', bad_lookup_symbols)
print('Inner.x in bad_lookup_symbols:', 'Inner.x' in bad_lookup_symbols)
""")


BadLookup symbols: ['inner_inst.x', 'y', 'Inner.x']
Inner.x in bad_lookup_symbols: True


### 4.12 `tree.ModelicaSemanticError` — self-extends

MLS §7.1.3 prohibits a class from extending itself directly or transitively.

In [43]:
# New: ModelicaSemanticError for self-extends
src = "model A  extends A;  Real x = 1.0;  end A;"
ast_tree = parser.parse(src)
try:
    tree.flatten(ast_tree, ast.ComponentRef(name="A"))
except tree.ModelicaSemanticError as exc:
    print(type(exc).__name__, ":", exc)

ModelicaSemanticError : Cannot extend class 'A' with itself


**In pymoca 0.11.2:** self-extends is also detected and a generic Python `Exception` is
raised. Other semantic errors (such as redeclaring a non-replaceable element, or
extending a `replaceable` class) are **silently ignored** — the model flattens as if the
invalid construct were permitted.  The new code's `ModelicaSemanticError` coverage is
broader.

In [44]:
run_old(f"""
import pymoca.parser as p
import pymoca.tree as t
import pymoca.ast as ast
tree_obj = p.parse('''{src}''')
try:
    t.flatten(tree_obj, ast.ComponentRef(name='A'))
except Exception as e:
    print(type(e).__name__, ':', str(e)[:120])
""")

Exception : Cannot extend class 'A' with itself


### 4.13 `tree.ModelicaSemanticError` — extending a `replaceable` class

MLS §7.3: you cannot `extends` a class that is declared `replaceable`; you must
`redeclare` it instead.

In [45]:
src = '''
model ExtendsRedeclareable
  replaceable model Base
    Real x = 1.0;
  end Base;
  extends Base;   // illegal: Base is replaceable — MLS 7.3
end ExtendsRedeclareable;
'''
ast_tree = parser.parse(src)
try:
    tree.flatten(ast_tree, ast.ComponentRef(name="ExtendsRedeclareable"))
except tree.ModelicaSemanticError as exc:
    print(type(exc).__name__, ":", exc)

ModelicaSemanticError : In ExtendsRedeclareable extends Base, Base and parents cannot be replaceable


**In pymoca 0.11.2:** extending a `replaceable` class is silently accepted.

In [46]:
run_old(f"""
from pymoca import parser, tree, ast
src = '''{src}'''
tree_obj = parser.parse(src)
try:
    result = tree.flatten(tree_obj, ast.ComponentRef(name='ExtendsRedeclareable'))
    print('No error raised — silently accepted')
except Exception as e:
    print(type(e).__name__, ':', str(e)[:120])
""")

No error raised — silently accepted


### 4.14 `tree.ModelicaSemanticError` — redeclaring a non-replaceable element

MLS §7.3: `redeclare` is only valid on elements declared `replaceable`.
In `RedeclareNonReplaceable.mo`, model `E` attempts to redeclare `D.C`
which is not declared `replaceable`.

In [47]:
ast_tree = parser.parse_file(MODEL_DIR / "RedeclareNonReplaceable.mo")
try:
    # The error fires at instantiation time (before flattening)
    tree.instantiate(ast_tree, "E")
except tree.ModelicaSemanticError as exc:
    print(type(exc).__name__, ":", exc)

ModelicaSemanticError : Redeclaring D.C that is not replaceable


**In pymoca 0.11.2:** `redeclare` on a non-`replaceable` element is silently
accepted — the model flattens as if the invalid redeclare were permitted.


In [48]:
path = str(MODEL_DIR / "RedeclareNonReplaceable.mo")
run_old(f"""
from pymoca import parser, tree, ast
with open(r'{path}') as f:
    src = f.read()
tree_obj = parser.parse(src)
try:
    flat_tree = tree.flatten(tree_obj, ast.ComponentRef(name='E'))
    print('No error raised — redeclaring non-replaceable silently accepted')
    print('classes:', list(flat_tree.classes))
except Exception as e:
    print(type(e).__name__, ':', str(e)[:120])
""")


No error raised — redeclaring non-replaceable silently accepted
classes: ['E']
